# 03 — Download DB-COMP source documents

## Purpose

This notebook retrieves European Commission competition-law documents from **DB-COMP** and stores them as one of the raw source collections used to construct the dataset.

The task of this notebook is deliberately limited. It:

1. queries DB-COMP for **Agreement** and **Dominance** records;
2. stores the returned search pages as raw HTML;
3. extracts a small amount of source metadata;
4. downloads the linked PDF files using the filenames supplied by DB-COMP; and
5. writes a compact manifest showing what was found, whether the file is available locally, and where it is stored.

No substantive document classification is carried out here. Text extraction, screening, harmonisation with the other Commission sources, and construction of the final dataset are handled in later stages of the pipeline.

## Source: DB-COMP

[DB-COMP](https://db-comp.eu/) is an independent academic database of European Commission competition-law decisional practice. It provides full-text search and contains English-language material with substantive legal content dating back to 1964. Its coverage includes agreements, abuse of dominance, mergers and the Digital Markets Act (DMA).

DB-COMP is particularly useful for this project because it provides a curated source for older Commission decisional practice that is not fully represented in the Commission's current online competition case database. DB-COMP is not affiliated with the European Commission.

The database used for this extraction was last updated in **June 2026**. For the present antitrust dataset, the query is restricted to **Agreement** and **Dominance** records and to the period from January 1964 to June 2026. Merger and DMA records are therefore outside the scope of this notebook.

More information on the source is available at [db-comp.eu/about-this-site](https://db-comp.eu/about-this-site).

## Raw-data principle

The HTML search pages and PDF documents are kept as close to the source as possible. In particular, PDF filenames are preserved rather than renamed into an internal naming convention.

The manifest adds only the metadata needed to identify and trace each document. Paths stored in the manifest are **relative to the repository root** (for example `data/raw/com_dbcomp/files/...`) so that the output does not contain usernames, drive letters or machine-specific directory structures.

A note on dates: `dbcomp_date` is the date displayed by DB-COMP for the search result. It is **not assumed to be the formal Commission decision date**; the latter is recovered from the document itself in a later processing step.

## 1. Imports and project paths

In [1]:
import json
import random
import re
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlencode, urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup
from IPython.display import display
from tqdm.auto import tqdm


def find_project_root(start=None):
    """Locate the ec_cjeu repository root from code/01_commission."""
    start = Path(start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "code" / "01_commission").exists():
            return candidate

    raise RuntimeError(
        "Could not locate the project root. "
        "Run this notebook from within the ec_cjeu repository."
    )


PROJECT_ROOT = find_project_root()

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "com_dbcomp"
HTML_DIR = RAW_DIR / "html"
FILES_DIR = RAW_DIR / "files"
LOG_DIR = PROJECT_ROOT / "logs" / "com_dbcomp"
OUTPUT_DIR = PROJECT_ROOT / "output" / "com_dbcomp"

for directory in (HTML_DIR, FILES_DIR, LOG_DIR, OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

DOWNLOAD_LOG_PATH = LOG_DIR / "db_comp_download.jsonl"
MANIFEST_PATH = OUTPUT_DIR / "db_comp_manifest.csv"

print("Raw HTML:  data/raw/com_dbcomp/html/")
print("Raw PDFs:  data/raw/com_dbcomp/files/")
print("Log:       logs/com_dbcomp/db_comp_download.jsonl")
print("Manifest:  output/com_dbcomp/db_comp_manifest.csv")

Raw HTML:  data/raw/com_dbcomp/html/
Raw PDFs:  data/raw/com_dbcomp/files/
Log:       logs/com_dbcomp/db_comp_download.jsonl
Manifest:  output/com_dbcomp/db_comp_manifest.csv


## 2. DB-COMP query and download settings

The query below reproduces the source scope used for this part of the dataset. DB-COMP's internal case-type values `1` and `2` correspond to **Agreement** and **Dominance** in the query used here.

Raw HTML is reused when it is already present. Set `REFRESH_HTML = True` only when intentionally creating a new source snapshot. Existing valid PDF files are also reused rather than downloaded again.

In [2]:
BASE_URL = "https://db-comp.eu/"

SEARCH_CONFIG = {
    "slotName": "Slot_Main_3",
    "Search_by_value": "details",
    "Document_type_values": "1,2",   # Agreement, Dominance
    "Month_from_value": "1",
    "Year_from_value": "1964",
    "Month_to_value": "6",
    "Year_to_value": "2026",
    "Mysql_operator_value": "none",
    "Sort_results_value": "relevance",
}

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 Academic research scraper for non-commercial "
        "EU competition-law dataset construction"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}

REFRESH_HTML = False
DOWNLOAD_FILES = True
OVERWRITE_FILES = False

REQUEST_TIMEOUT = 30
PAGE_SLEEP_SECONDS = (1.0, 2.5)
DOWNLOAD_SLEEP_SECONDS = (0.3, 0.8)

## 3. Helpers for retrieving and parsing DB-COMP

In [3]:
def clean_text(value):
    if value is None:
        return ""
    return re.sub(r"\s+", " ", str(value).replace("\xa0", " ")).strip()


def sleep_random(bounds):
    low, high = bounds
    if high > 0:
        time.sleep(random.uniform(low, high))


def make_page_url(page):
    params = dict(SEARCH_CONFIG)
    params["page"] = str(page)
    return BASE_URL + "?" + urlencode(params)


def fetch_page(session, page):
    """Return one DB-COMP result page as raw bytes and keep the raw response locally."""
    raw_path = HTML_DIR / f"page_{page:04d}.html"

    if raw_path.exists() and not REFRESH_HTML:
        return raw_path.read_bytes()

    response = session.get(
        make_page_url(page),
        headers=HEADERS,
        timeout=REQUEST_TIMEOUT,
    )
    response.raise_for_status()

    # Save source bytes directly rather than decoding and re-encoding them.
    raw_path.write_bytes(response.content)
    return response.content


def make_soup(html_bytes):
    # Passing bytes lets BeautifulSoup inspect the source encoding itself.
    return BeautifulSoup(html_bytes, "html.parser")


def parse_total_results(html_bytes):
    text = make_soup(html_bytes).get_text("\n", strip=True)
    match = re.search(r"(\d[\d,]*)\s+results", text, flags=re.IGNORECASE)
    if not match:
        return None
    return int(match.group(1).replace(",", ""))


def parse_last_page(html_bytes):
    soup = make_soup(html_bytes)

    for link in soup.find_all("a", href=True):
        if clean_text(link.get_text(" ", strip=True)).lower() == "last":
            match = re.search(r"[?&]page=(\d+)", link["href"])
            if match:
                return int(match.group(1))

    # A normal pagination bar is also sufficient; there is no hard-coded
    # fallback page count.
    page_numbers = []
    for link in soup.find_all("a", href=True):
        match = re.search(r"[?&]page=(\d+)", link["href"])
        if match:
            page_numbers.append(int(match.group(1)))

    return max(page_numbers) if page_numbers else None


def parse_case_type_and_date(text):
    """Read DB-COMP's displayed case type(s) and date without calling it a decision date."""
    match = re.search(
        r"((?:Agreement|Dominance)(?:\s*,\s*(?:Agreement|Dominance))*)"
        r"\s+Date:\s*([0-9]{1,2}\s+[A-Za-z]{3}\s+[0-9]{4})",
        text,
        flags=re.IGNORECASE,
    )
    if not match:
        return "", pd.NaT

    types = [part.strip().title() for part in match.group(1).split(",")]
    case_type = ";".join(dict.fromkeys(types))
    dbcomp_date = pd.to_datetime(match.group(2), format="%d %b %Y", errors="coerce")

    return case_type, dbcomp_date


def extract_case_number_from_dbcomp_text(text, document_name):
    """Use the case number shown at the start of the DB-COMP result when available."""
    if document_name and document_name in text:
        prefix = text.split(document_name, 1)[0].strip()
    else:
        prefix = text

    numbers = re.findall(r"(?<!\d)(\d{2,6})(?!\d)", prefix)
    if numbers:
        return ";".join(dict.fromkeys(numbers))
    return ""


def extract_case_number_from_filename(file_name):
    """Conservative fallback used only when DB-COMP does not display a case number."""
    stem = Path(file_name).stem

    match = re.match(
        r"^(?:AT[._-]?)?(\d{4,6}(?:[-_]\d{4,6})*)",
        stem,
        flags=re.IGNORECASE,
    )
    if not match:
        return ""

    numbers = re.split(r"[-_]", match.group(1))
    return ";".join(dict.fromkeys(numbers))


def extract_dbcomp_document_id(file_url):
    """Return DB-COMP's own numeric document ID where the URL contains one."""
    match = re.search(r"/document_(\d+)\.download(?:$|[?#])", file_url or "")
    return match.group(1) if match else pd.NA


def parse_result_block(block):
    text = clean_text(block.get_text(" ", strip=True))

    links = []
    for link in block.find_all("a", href=True):
        links.append(
            {
                "text": clean_text(link.get_text(" ", strip=True)),
                "href": urljoin(BASE_URL, link["href"]),
            }
        )

    download_links = [link for link in links if ".download" in link["href"]]
    if not download_links:
        return None

    file_url = download_links[0]["href"]

    document_name = next(
        (link["text"] for link in download_links if link["text"]),
        "",
    )
    if not document_name:
        document_name = next((link["text"] for link in links if link["text"]), "")

    # DB-COMP displays the source filename in the result text. Keep it as given.
    file_match = re.search(r"(\S+\.pdf)\b", text, flags=re.IGNORECASE)
    file_name = file_match.group(1) if file_match else ""

    case_number = extract_case_number_from_dbcomp_text(text, document_name)
    case_number_source = "dbcomp" if case_number else ""

    if not case_number and file_name:
        case_number = extract_case_number_from_filename(file_name)
        case_number_source = "filename" if case_number else ""

    case_type, dbcomp_date = parse_case_type_and_date(text)

    return {
        "source": "db-comp.eu",
        "dbcomp_document_id": extract_dbcomp_document_id(file_url),
        "case_number": case_number or pd.NA,
        "case_number_source": case_number_source or pd.NA,
        "case_type": case_type or pd.NA,
        "dbcomp_date": dbcomp_date,
        "document_name": document_name or pd.NA,
        "file_url": file_url,
        "file_name": file_name or pd.NA,
    }


def parse_results(html_bytes):
    soup = make_soup(html_bytes)
    blocks = soup.select("div.result")

    if not blocks:
        raise RuntimeError(
            "No DB-COMP result blocks were found. "
            "The source HTML structure may have changed."
        )

    rows = []
    for block in blocks:
        row = parse_result_block(block)
        if row is not None:
            rows.append(row)

    return rows

## 4. Build the source manifest

The first result page provides the reported number of search results and the pagination. The notebook stops if these cannot be determined reliably; it does not guess a fixed number of pages.

The resulting manifest contains one row per DB-COMP download URL. DB-COMP document IDs are retained only when they are actually present in the source URL. Missing source IDs are left missing rather than replaced with synthetic values.

In [4]:
def build_manifest():
    session = requests.Session()

    first_page = fetch_page(session, page=1)
    total_results = parse_total_results(first_page)
    first_rows = parse_results(first_page)
    last_page = parse_last_page(first_page)

    if total_results is None:
        raise RuntimeError("Could not read DB-COMP's reported result count.")

    if last_page is None:
        if total_results == len(first_rows):
            last_page = 1
        else:
            raise RuntimeError(
                "Could not determine DB-COMP pagination. "
                "Stopping rather than using a hard-coded fallback."
            )

    print(f"DB-COMP reports {total_results:,} records across {last_page:,} page(s).")

    rows = list(first_rows)

    for page in tqdm(range(2, last_page + 1), desc="Retrieving DB-COMP pages"):
        html = fetch_page(session, page)
        rows.extend(parse_results(html))
        sleep_random(PAGE_SLEEP_SECONDS)

    manifest = pd.DataFrame(rows)

    if manifest["file_url"].duplicated().any():
        duplicate_urls = manifest.loc[
            manifest["file_url"].duplicated(keep=False), "file_url"
        ].unique()
        raise RuntimeError(
            f"Duplicate DB-COMP file URLs found ({len(duplicate_urls)} unique duplicates)."
        )

    if len(manifest) != total_results:
        raise RuntimeError(
            f"DB-COMP reported {total_results:,} results, "
            f"but {len(manifest):,} records were parsed."
        )

    manifest["dbcomp_date"] = pd.to_datetime(
        manifest["dbcomp_date"], errors="coerce"
    ).dt.strftime("%Y-%m-%d")

    return manifest.reset_index(drop=True)


manifest = build_manifest()
display(manifest.head())

DB-COMP reports 837 records across 42 page(s).


Retrieving DB-COMP pages:   0%|          | 0/41 [00:00<?, ?it/s]

,source,dbcomp_document_id,case_number,case_number_source,case_type,dbcomp_date,document_name,file_url,file_name
0,db-comp.eu,1061,93,dbcomp,Agreement,1969-03-13,European Machine Tool Exhibitions (EEMO),https://db-comp.eu/document_1061.download,CELEX-31979D0037-EN-TXT.pdf
1,db-comp.eu,1064,399904,dbcomp,Agreement,2016-12-12,Rechargeable battery,https://db-comp.eu/document_1064.download,399904-rechargeable-battery-(Dec-2016).pdf
2,db-comp.eu,1065,40481,dbcomp,Agreement,2019-03-05,Occupant Safety Systems (II) supplied to the V...,https://db-comp.eu/document_1065.download,40481-Occupant-Safety-Systems.pdf
3,db-comp.eu,1066,40360,dbcomp,Agreement,2017-01-19,Production and distribution of audiobooks,https://db-comp.eu/document_1066.download,40360-production-and-distribution-of-audiobook...
4,db-comp.eu,1067,40291,dbcomp,Agreement,2016-01-21,Aquatrend,https://db-comp.eu/document_1067.download,40291-Aquatrend-(Jan-2016).pdf


## 5. Download the source PDFs

PDFs are stored under `data/raw/com_dbcomp/files/` using the filename displayed by DB-COMP.

Two mechanisms are used during downloading:

- `tqdm` provides the live progress bar while the notebook is running;
- `logs/com_dbcomp/db_comp_download.jsonl` records each download attempt immediately so that progress survives an interrupted kernel or stopped download.

The JSONL log is an operational download tracker rather than a publication-facing dataset file. On restart, the notebook checks both the tracker and the files on disk. A valid PDF on disk is treated as authoritative: if the tracker says a file was downloaded but the PDF is missing, the file is downloaded again.

The manifest records:

- `downloaded`: whether a valid PDF is present locally;
- `local_path`: the repository-relative path where the file is stored; and
- `download_error`: populated only when a requested download fails.

Repository-relative paths keep the manifest portable and avoid exposing local usernames, drive letters or absolute directory structures.

In [5]:
def local_pdf_path(file_name):
    if pd.isna(file_name) or not str(file_name).strip():
        return None

    # Preserve the source filename itself, but never allow a source-side
    # path component to escape the intended data directory.
    return FILES_DIR / Path(str(file_name)).name


def repository_relative(path):
    return path.relative_to(PROJECT_ROOT).as_posix()


def is_valid_pdf(path):
    if path is None or not path.exists() or path.stat().st_size == 0:
        return False

    with path.open("rb") as file:
        return file.read(5) == b"%PDF-"


def append_download_log(record):
    """Append one download event immediately to the JSONL tracker."""
    record = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        **record,
    }
    with DOWNLOAD_LOG_PATH.open("a", encoding="utf-8") as log_file:
        log_file.write(json.dumps(record, ensure_ascii=False) + "\n")


def read_download_log():
    """Return the latest JSONL record for each file URL."""
    latest = {}

    if not DOWNLOAD_LOG_PATH.exists():
        return latest

    with DOWNLOAD_LOG_PATH.open("r", encoding="utf-8") as log_file:
        for line in log_file:
            line = line.strip()
            if not line:
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                # A partially written final line should not prevent restart.
                continue

            file_url = record.get("file_url")
            if file_url:
                latest[file_url] = record

    return latest


def set_local_file_state(df):
    df = df.copy()

    local_paths = []
    downloaded = []

    for file_name in df["file_name"]:
        path = local_pdf_path(file_name)
        local_paths.append(repository_relative(path) if path else pd.NA)
        downloaded.append(is_valid_pdf(path))

    df["downloaded"] = downloaded
    df["local_path"] = local_paths

    if "download_error" not in df.columns:
        df["download_error"] = pd.NA

    return df


def download_documents(df):
    df = set_local_file_state(df)
    session = requests.Session()
    previous_log = read_download_log()

    pending = df.index[~df["downloaded"]].tolist()

    print(f"PDFs already present: {int(df['downloaded'].sum()):,}")
    print(f"PDFs to download:     {len(pending):,}")
    if previous_log:
        print(f"Tracked URLs in JSONL: {len(previous_log):,}")

    if not DOWNLOAD_FILES:
        return df

    for idx in tqdm(pending, desc="Downloading DB-COMP PDFs"):
        url = df.at[idx, "file_url"]
        file_name = df.at[idx, "file_name"]
        output_path = local_pdf_path(file_name)
        dbcomp_document_id = df.at[idx, "dbcomp_document_id"]

        if output_path is None:
            error = "Missing source filename"
            df.at[idx, "download_error"] = error

            append_download_log({
                "dbcomp_document_id": None if pd.isna(dbcomp_document_id) else str(dbcomp_document_id),
                "file_url": url,
                "file_name": None,
                "status": "failed",
                "local_path": None,
                "error": error,
            })
            continue

        relative_path = repository_relative(output_path)

        # The file on disk is authoritative. This also makes interrupted
        # downloads naturally resumable even if the previous kernel stopped.
        if is_valid_pdf(output_path) and not OVERWRITE_FILES:
            df.at[idx, "downloaded"] = True
            df.at[idx, "local_path"] = relative_path
            df.at[idx, "download_error"] = pd.NA

            append_download_log({
                "dbcomp_document_id": None if pd.isna(dbcomp_document_id) else str(dbcomp_document_id),
                "file_url": url,
                "file_name": str(file_name),
                "status": "already_present",
                "local_path": relative_path,
                "error": None,
            })
            continue

        try:
            response = session.get(
                url,
                headers=HEADERS,
                timeout=REQUEST_TIMEOUT,
            )
            response.raise_for_status()

            if not response.content.startswith(b"%PDF-"):
                raise ValueError("Response is not a PDF")

            output_path.write_bytes(response.content)

            if not is_valid_pdf(output_path):
                raise ValueError("Downloaded file failed PDF validation")

            df.at[idx, "downloaded"] = True
            df.at[idx, "local_path"] = relative_path
            df.at[idx, "download_error"] = pd.NA

            append_download_log({
                "dbcomp_document_id": None if pd.isna(dbcomp_document_id) else str(dbcomp_document_id),
                "file_url": url,
                "file_name": str(file_name),
                "status": "downloaded",
                "local_path": relative_path,
                "error": None,
            })

        except Exception as exc:
            error = str(exc)
            df.at[idx, "downloaded"] = False
            df.at[idx, "local_path"] = relative_path
            df.at[idx, "download_error"] = error

            append_download_log({
                "dbcomp_document_id": None if pd.isna(dbcomp_document_id) else str(dbcomp_document_id),
                "file_url": url,
                "file_name": str(file_name),
                "status": "failed",
                "local_path": relative_path,
                "error": error,
            })

        sleep_random(DOWNLOAD_SLEEP_SECONDS)

    return df


manifest = download_documents(manifest)

PDFs already present: 0
PDFs to download:     837


## 6. Save the compact manifest

The manifest is intentionally small. Raw HTML and raw PDFs provide the source-level record; the CSV only organises the fields needed to trace and process the collection.

`case_number_source` is retained because a small number of DB-COMP results do not display a case number directly. In those cases the notebook uses a conservative filename fallback. No separate audit columns are created for the two versions.

In [6]:
MANIFEST_COLUMNS = [
    "source",
    "dbcomp_document_id",
    "case_number",
    "case_number_source",
    "case_type",
    "dbcomp_date",
    "document_name",
    "file_url",
    "file_name",
    "downloaded",
    "local_path",
    "download_error",
]

manifest = manifest[MANIFEST_COLUMNS].copy()
manifest.to_csv(MANIFEST_PATH, index=False, encoding="utf-8")

print("Saved manifest: output/com_dbcomp/db_comp_manifest.csv")
display(manifest.head())

Saved manifest: output/com_dbcomp/db_comp_manifest.csv


,source,dbcomp_document_id,case_number,case_number_source,case_type,dbcomp_date,document_name,file_url,file_name,downloaded,local_path,download_error
0,db-comp.eu,1061,93,dbcomp,Agreement,1969-03-13,European Machine Tool Exhibitions (EEMO),https://db-comp.eu/document_1061.download,CELEX-31979D0037-EN-TXT.pdf,True,data/raw/com_dbcomp/files/CELEX-31979D0037-EN-...,<NA>
1,db-comp.eu,1064,399904,dbcomp,Agreement,2016-12-12,Rechargeable battery,https://db-comp.eu/document_1064.download,399904-rechargeable-battery-(Dec-2016).pdf,True,data/raw/com_dbcomp/files/399904-rechargeable-...,<NA>
2,db-comp.eu,1065,40481,dbcomp,Agreement,2019-03-05,Occupant Safety Systems (II) supplied to the V...,https://db-comp.eu/document_1065.download,40481-Occupant-Safety-Systems.pdf,True,data/raw/com_dbcomp/files/40481-Occupant-Safet...,<NA>
3,db-comp.eu,1066,40360,dbcomp,Agreement,2017-01-19,Production and distribution of audiobooks,https://db-comp.eu/document_1066.download,40360-production-and-distribution-of-audiobook...,True,data/raw/com_dbcomp/files/40360-production-and...,<NA>
4,db-comp.eu,1067,40291,dbcomp,Agreement,2016-01-21,Aquatrend,https://db-comp.eu/document_1067.download,40291-Aquatrend-(Jan-2016).pdf,True,data/raw/com_dbcomp/files/40291-Aquatrend-(Jan...,<NA>


## 7. Technical checks

These checks are intentionally limited to acquisition and traceability: whether the source records were parsed once, whether filenames and URLs are present, whether local paths are portable, and whether the expected PDFs are available.

They are not a substantive validation of document content.

In [7]:
missing_source_ids = int(manifest["dbcomp_document_id"].isna().sum())
missing_case_numbers = int(manifest["case_number"].isna().sum())
filename_inferred = int(manifest["case_number_source"].eq("filename").sum())
missing_file_names = int(manifest["file_name"].isna().sum())
duplicate_urls = int(manifest["file_url"].duplicated().sum())
downloaded_count = int(manifest["downloaded"].sum())
failed_count = int((~manifest["downloaded"]).sum())

absolute_paths = manifest["local_path"].dropna().astype(str).map(
    lambda value: Path(value).is_absolute()
    or bool(re.match(r"^[A-Za-z]:[/\\\\]", value))
)
absolute_path_count = int(absolute_paths.sum())

print(f"Manifest records:        {len(manifest):,}")
print(f"PDFs available locally:  {downloaded_count:,} / {len(manifest):,}")
print(f"PDFs unavailable:        {failed_count:,}")
print(f"Missing DB-COMP IDs:     {missing_source_ids:,}")
print(f"Case numbers inferred:   {filename_inferred:,}")
print(f"Missing case numbers:    {missing_case_numbers:,}")
print(f"Missing filenames:       {missing_file_names:,}")
print(f"Duplicate file URLs:     {duplicate_urls:,}")
print(f"Absolute local paths:    {absolute_path_count:,}")
print(f"Download tracker:        logs/com_dbcomp/db_comp_download.jsonl")

assert duplicate_urls == 0, "Duplicate DB-COMP file URLs remain in the manifest."
assert missing_file_names == 0, "Some DB-COMP records have no source filename."
assert absolute_path_count == 0, "Machine-specific absolute paths remain in the manifest."

if DOWNLOAD_FILES and failed_count:
    display(
        manifest.loc[
            ~manifest["downloaded"],
            ["case_number", "document_name", "file_url", "download_error"],
        ]
    )
    raise RuntimeError(
        f"{failed_count} DB-COMP PDF(s) are not available locally. "
        "See the displayed failed records above."
    )

Manifest records:        837
PDFs available locally:  837 / 837
PDFs unavailable:        0
Missing DB-COMP IDs:     6
Case numbers inferred:   0
Missing case numbers:    0
Missing filenames:       0
Duplicate file URLs:     0
Absolute local paths:    0
Download tracker:        logs/com_dbcomp/db_comp_download.jsonl


## Output

The notebook produces one publication-facing manifest:

`output/com_dbcomp/db_comp_manifest.csv`

and preserves the raw source collection under:

- `data/raw/com_dbcomp/html/`
- `data/raw/com_dbcomp/files/`

Download progress is recorded separately in:

- `logs/com_dbcomp/db_comp_download.jsonl`

The next pipeline step can use `case_number`, `file_name` and `local_path` to process the downloaded documents without relying on machine-specific paths.